# Proyecto Final - Agente RL para Connect-4
**Curso:** Fundamentos de Inteligencia Artificial - Universidad de La Sabana, 2026.1  
**Rama:** Martin-Jerez

## Arquitectura del agente

El `QLearningAgent` combina dos ideas del curso:

1. **Q-Learning offline (Diapo 12):** Auto-juego con truco bipolar aprende pesos `w` para una funcion de evaluacion `V(s) = w * psi(s)`.
2. **Alpha-Beta Minimax depth=6:** Durante el juego, busca 6 niveles usando `V(s)` como heuristica de hoja - permite detectar forks y tacticas sin programarlas.


In [ ]:
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from rl_agent import QLearningAgent
from mcts_random import MCTSAgentRandom

ROWS, COLS = 6, 7

def apply_move(board, col, player):
    new_board = board.copy()
    for r in range(ROWS - 1, -1, -1):
        if new_board[r, col] == 0:
            new_board[r, col] = player
            break
    return new_board

def check_winner(board):
    for r in range(ROWS):
        for c in range(COLS):
            p = board[r, c]
            if p == 0: continue
            if c+3<COLS and all(board[r,c+i]==p for i in range(4)): return p
            if r+3<ROWS and all(board[r+i,c]==p for i in range(4)): return p
            if r+3<ROWS and c+3<COLS and all(board[r+i,c+i]==p for i in range(4)): return p
            if r+3<ROWS and c-3>=0  and all(board[r+i,c-i]==p for i in range(4)): return p
    return 0

def random_act(board):
    free = [c for c in range(COLS) if board[0, c] == 0]
    return random.choice(free) if free else 3

def play_game(agent_a, agent_b, a_is_minus1=True):
    board = np.zeros((ROWS, COLS), dtype=int)
    a_player = -1 if a_is_minus1 else 1
    current = -1
    while True:
        free = [c for c in range(COLS) if board[0, c] == 0]
        if not free: return 0
        act_fn = (agent_a if callable(agent_a) else agent_a.act) if current == a_player \
                 else (agent_b if callable(agent_b) else agent_b.act)
        col = act_fn(board)
        board = apply_move(board, col, current)
        w = check_winner(board)
        if w != 0: return 1 if w == a_player else -1
        current = -current

def run_series(agent, opponent, n_games=100):
    wins = draws = losses = 0
    for i in range(n_games):
        r = play_game(agent, opponent, a_is_minus1=(i % 2 == 0))
        if r == 1: wins += 1
        elif r == 0: draws += 1
        else: losses += 1
    return wins, draws, losses

print('Utilidades cargadas.')


## Experimento 1: Impacto de `n_episodes` vs. agente aleatorio

Entrenamos el `QLearningAgent` con distintos presupuestos de episodios y medimos la tasa de victorias contra un agente que elige columnas al azar.

In [ ]:
episode_values = [100, 500, 1000, 2000, 5000]
win_rates = []

for n_ep in episode_values:
    print(f'Entrenando con n_episodes={n_ep}...')
    agent = QLearningAgent(player=-1, n_episodes=n_ep)
    agent.mount()
    wins, draws, losses = run_series(agent, random_act, n_games=100)
    wr = wins / 100
    win_rates.append(wr)
    print(f'  W={wins} D={draws} L={losses}  (win rate={wr:.2%})')

plt.figure(figsize=(8, 4))
plt.plot(episode_values, [w*100 for w in win_rates], marker='o', linewidth=2)
plt.axhline(50, color='red', linestyle='--', label='Umbral 50%')
plt.xlabel('n_episodes (presupuesto de entrenamiento)')
plt.ylabel('Tasa de victorias (%)')
plt.title('QLearningAgent vs. Agente Aleatorio')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Experimento 2: Auto-desempeño – RL fuerte vs. RL débil

Enfrentamos el agente con 5000 episodios contra uno entrenado con solo 500.

In [ ]:
print('Entrenando agente fuerte (5000 ep)...')
strong = QLearningAgent(player=-1, n_episodes=5000)
strong.mount()

print('Entrenando agente débil (500 ep)...')
weak = QLearningAgent(player=1, n_episodes=500)
weak.mount()

wins, draws, losses = run_series(strong, weak.act, n_games=100)
print(f'Strong vs Weak → W={wins} D={draws} L={losses}')

labels = ['Victorias\n(strong)', 'Empates', 'Derrotas\n(strong)']
values = [wins, draws, losses]
colors = ['#4CAF50', '#FFC107', '#F44336']
plt.figure(figsize=(6, 4))
plt.bar(labels, values, color=colors)
plt.title('RL-5000ep vs RL-500ep (100 partidas)')
plt.ylabel('Número de partidas')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## Experimento 3: QLearningAgent (5000 ep) vs. MCTSAgentRandom (200 sims)

Comparamos el aprendizaje offline contra la búsqueda online del agente base del `master`.

In [ ]:
print('Entrenando QLearningAgent (5000 ep, depth=6)...')
t0 = time.time()
ql_agent = QLearningAgent(player=-1, n_episodes=5000, search_depth=6)
ql_agent.mount()
print(f'Entrenamiento: {time.time()-t0:.1f}s  |  Pesos: {ql_agent.weights.round(3)}')

mcts_agent = MCTSAgentRandom(player=1, num_simulations=200)
mcts_agent.mount()

print('Jugando 50 partidas QL vs MCTS-200...')
wins, draws, losses = run_series(ql_agent, mcts_agent.act, n_games=50)
print(f'QL vs MCTS-200 -> W={wins} D={draws} L={losses}  (win rate={wins/50:.0%})')

labels = ['QL gana', 'Empate', 'MCTS gana']
plt.figure(figsize=(6, 4))
plt.bar(labels, [wins, draws, losses], color=['#2196F3', '#FFC107', '#FF5722'])
plt.title('QLearningAgent (5000 ep, depth=6) vs MCTSAgentRandom (200 sims)\n50 partidas')
plt.ylabel('Numero de partidas'); plt.grid(axis='y'); plt.tight_layout(); plt.show()


## Experimento 4: Impacto de la profundidad Alpha-Beta

La profundidad es la variable de configuracion del componente de busqueda online.

In [ ]:
print('Entrenando agente base (5000 ep)...')
base = QLearningAgent(player=-1, n_episodes=5000)
base.mount()
base_weights = base.weights.copy()

mcts = MCTSAgentRandom(player=1, num_simulations=200)
mcts.mount()

depths = [0, 2, 4, 6]
depth_wins = []

for d in depths:
    agent_d = QLearningAgent(player=-1, n_episodes=1, search_depth=d)
    agent_d.weights = base_weights.copy()
    w, dr, l = run_series(agent_d, mcts.act, n_games=50)
    depth_wins.append(w)
    print(f'depth={d}: W={w} D={dr} L={l}  ({w*2}%)')

plt.figure(figsize=(7, 4))
plt.plot(depths, [w*2 for w in depth_wins], marker='s', linewidth=2, color='#2196F3')
plt.axhline(50, color='red', linestyle='--', label='50%')
plt.xlabel('Profundidad Alpha-Beta')
plt.ylabel('Win rate vs MCTS-200 (%)')
plt.title('Profundidad de busqueda vs MCTSAgentRandom (200 sims)')
plt.xticks(depths); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


## Conclusiones

### Por que Alpha-Beta + Q-Learning supera al lookup plano?

El agente v1 (solo Q-lookup) tenia **cero lookahead** y perdia contra MCTS-200 (6 victorias de 50).
La version mejorada combina:
- **Funcion de evaluacion aprendida** (`V(s) = w * psi(s)`) que captura patrones estrategicos
- **Alpha-Beta profundidad 6** que explota esa evaluacion para razonar tacticamente

### Tabla comparativa

| Aspecto | MCTSAgentRandom | QLearning v1 | **QLearning v2** |
|---------|-----------------|--------------|------------------|
| Paradigma | Busqueda online | Aprendizaje offline | **RL + busqueda** |
| Lookahead | Si (MCTS) | No | **Si (Alpha-Beta d=6)** |
| Evaluacion | Rollouts aleatorios | Pesos aprendidos | **Pesos aprendidos** |
| Variable | # simulaciones | # episodios | **# episodios + depth** |
| Fundamento | Diapo 13 | Diapo 12 | **Diapo 12 + 13** |

### Observaciones

1. **`n_episodes`:** mas auto-juego mejora V(s), analogo a mas simulaciones en MCTS.
2. **Truco bipolar:** propagar negando el valor cada turno (`r = -gamma*r`) permite aprender una sola Q-function valida para ambos jugadores (Diapo 12).
3. **Depth:** a profundidad 0 el agente pierde; a depth 6 detecta tacticas como forks implicitamente.
4. **Limitacion:** aproximacion lineal con 8 features; DQN seria el siguiente paso natural.
